In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

In [2]:
api_key = "YOUR_KEY"

In [3]:
url = 'https://www.alphavantage.co/query'

In [4]:
params = {
    'function': 'NEWS_SENTIMENT',
    'tickers': 'JPM',
    'time_from': '20240101T0000',
    'time_to': '20240401T0000',
    'limit': 50,
    'apikey': api_key
}

response = requests.get(url, params=params)
data = response.json()
print(data.keys())

dict_keys(['items', 'sentiment_score_definition', 'relevance_score_definition', 'feed'])


In [5]:
print(data['items'])
print(data['feed'][0])

50
{'title': 'Banc of California: Post-Merger Optimism', 'url': 'https://labusinessjournal.com/special-reports/banc-of-california-post-merger-optimism/', 'time_published': '20240401T000000', 'authors': ['James Brock'], 'summary': "Banc of California CEO Jared Wolff discusses the successful and rapid integration following its 2023 merger with PacWest, which created a $38 billion entity. Wolff highlights the bank's strategy in bridging capital for portfolio companies and managing its commercial real estate exposure, particularly to office space. He also shares his optimistic outlook on the Southern California economy's resilience despite recent regional banking industry shifts and transaction slowdowns.", 'banner_image': None, 'source': 'Los Angeles Business Journal', 'category_within_source': 'General', 'source_domain': 'Los Angeles Business Journal', 'topics': [{'topic': 'mergers_and_acquisitions', 'relevance_score': '1.000000'}, {'topic': 'finance', 'relevance_score': '1.000000'}, {'t

In [6]:
import json
print(json.dumps(data['feed'][0], indent=2))

{
  "title": "Banc of California: Post-Merger Optimism",
  "url": "https://labusinessjournal.com/special-reports/banc-of-california-post-merger-optimism/",
  "time_published": "20240401T000000",
  "authors": [
    "James Brock"
  ],
  "summary": "Banc of California CEO Jared Wolff discusses the successful and rapid integration following its 2023 merger with PacWest, which created a $38 billion entity. Wolff highlights the bank's strategy in bridging capital for portfolio companies and managing its commercial real estate exposure, particularly to office space. He also shares his optimistic outlook on the Southern California economy's resilience despite recent regional banking industry shifts and transaction slowdowns.",
  "banner_image": null,
  "source": "Los Angeles Business Journal",
  "category_within_source": "General",
  "source_domain": "Los Angeles Business Journal",
  "topics": [
    {
      "topic": "mergers_and_acquisitions",
      "relevance_score": "1.000000"
    },
    {
 

In [7]:
records = []

for article in data['feed']:
    date = article['time_published'][:8]
    for ticker_info in article['ticker_sentiment']:
        if ticker_info['ticker'] == 'JPM':
            relevance = float(ticker_info['relevance_score'])
            sentiment = float(ticker_info['ticker_sentiment_score'])
            weighted = sentiment * relevance
            records.append({'date': date, 'weighted_sentiment': weighted})

sentiment_df = pd.DataFrame(records)
print(sentiment_df.shape)
print(sentiment_df.head())

(50, 2)
       date  weighted_sentiment
0  20240401           -0.064813
1  20240329            0.269141
2  20240328            0.064771
3  20240328            0.140170
4  20240328            0.020734


In [8]:
params_old = {
    'function': 'NEWS_SENTIMENT',
    'tickers': 'JPM',
    'time_from': '20180101T0000',
    'time_to': '20180401T0000',
    'limit': 50,
    'apikey': api_key
}

response_old = requests.get(url, params=params_old)
data_old = response_old.json()
print(data_old.get('items', 'no items key'))
print(data_old.get('feed', 'no feed key')[:1] if 'feed' in data_old else data_old)

no items key
{'Information': 'Thank you for using Alpha Vantage! Please consider spreading out your free API requests more sparingly (1 request per second). You may subscribe to any of the premium plans at https://www.alphavantage.co/premium/ to lift the free key rate limit (25 requests per day), raise the per-second burst limit, and instantly unlock all premium endpoints'}


In [9]:
print(json.dumps(data['feed'][0], indent=2))

{
  "title": "Banc of California: Post-Merger Optimism",
  "url": "https://labusinessjournal.com/special-reports/banc-of-california-post-merger-optimism/",
  "time_published": "20240401T000000",
  "authors": [
    "James Brock"
  ],
  "summary": "Banc of California CEO Jared Wolff discusses the successful and rapid integration following its 2023 merger with PacWest, which created a $38 billion entity. Wolff highlights the bank's strategy in bridging capital for portfolio companies and managing its commercial real estate exposure, particularly to office space. He also shares his optimistic outlook on the Southern California economy's resilience despite recent regional banking industry shifts and transaction slowdowns.",
  "banner_image": null,
  "source": "Los Angeles Business Journal",
  "category_within_source": "General",
  "source_domain": "Los Angeles Business Journal",
  "topics": [
    {
      "topic": "mergers_and_acquisitions",
      "relevance_score": "1.000000"
    },
    {
 

In [10]:
date_ranges = [
    ('20180101T0000', '20180701T0000'),
    ('20180701T0000', '20190101T0000'),
    ('20190101T0000', '20190701T0000'),
    ('20190701T0000', '20200101T0000'),
    ('20200101T0000', '20200701T0000'),
    ('20200701T0000', '20210101T0000'),
    ('20210101T0000', '20210701T0000'),
    ('20210701T0000', '20220101T0000'),
    ('20220101T0000', '20220701T0000'),
    ('20220701T0000', '20230101T0000'),
    ('20230101T0000', '20230701T0000'),
    ('20230701T0000', '20240101T0000'),
    ('20240101T0000', '20240701T0000'),
    ('20240701T0000', '2025101T0000'),


]

In [11]:
import time

all_records = []

for start, end in date_ranges:
    params = {
        'function': 'NEWS_SENTIMENT',
        'tickers': 'JPM',
        'time_from': start,
        'time_to': end,
        'limit': 1000,
        'apikey': api_key
    }
    response = requests.get(url, params=params)
    data = response.json()

    if 'feed' not in data:
        print(f"No feed for {start} to {end}")
        continue

    for article in data['feed']:
        date = article['time_published'][:8]
        for ticker_info in article['ticker_sentiment']:
            if ticker_info['ticker'] == 'JPM':
                relevance = float(ticker_info['relevance_score'])
                sentiment = float(ticker_info['ticker_sentiment_score'])
                weighted = sentiment * relevance
                all_records.append({'date': date, 'weighted_sentiment': weighted})

    time.sleep(15)
    print(f"Done: {start} to {end}, total records so far: {len(all_records)}")

No feed for 20180101T0000 to 20180701T0000
No feed for 20180701T0000 to 20190101T0000
Done: 20190101T0000 to 20190701T0000, total records so far: 194
Done: 20190701T0000 to 20200101T0000, total records so far: 392
Done: 20200101T0000 to 20200701T0000, total records so far: 573
Done: 20200701T0000 to 20210101T0000, total records so far: 1039
Done: 20210101T0000 to 20210701T0000, total records so far: 1637
Done: 20210701T0000 to 20220101T0000, total records so far: 2190
Done: 20220101T0000 to 20220701T0000, total records so far: 2708
Done: 20220701T0000 to 20230101T0000, total records so far: 3268
Done: 20230101T0000 to 20230701T0000, total records so far: 3690
Done: 20230701T0000 to 20240101T0000, total records so far: 4061
Done: 20240101T0000 to 20240701T0000, total records so far: 4536
Done: 20240701T0000 to 2025101T0000, total records so far: 5537


In [12]:
sentiment_df = pd.DataFrame(all_records)
sentiment_df.to_csv('jpm_news_sentiment_raw.csv', index=False)
print(sentiment_df.shape)
print(sentiment_df.head())

(5537, 2)
       date  weighted_sentiment
0  20190628            0.077050
1  20190626            0.084559
2  20190626            0.012080
3  20190626            0.426255
4  20190625            0.025971


In [13]:
sentiment_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5537 entries, 0 to 5536
Data columns (total 2 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   date                5537 non-null   object 
 1   weighted_sentiment  5537 non-null   float64
dtypes: float64(1), object(1)
memory usage: 86.6+ KB


In [14]:
sentiment_df['date'] = pd.to_datetime(sentiment_df['date'], errors='coerce')

In [15]:
sentiment_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5537 entries, 0 to 5536
Data columns (total 2 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   date                5537 non-null   datetime64[ns]
 1   weighted_sentiment  5537 non-null   float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 86.6 KB


In [16]:
daily_sentiment = sentiment_df.groupby('date')['weighted_sentiment'].mean().reset_index()
daily_sentiment.shape

(1580, 2)

In [17]:
print(daily_sentiment.head())

        date  weighted_sentiment
0 2019-01-01           -0.056234
1 2019-01-03            0.240760
2 2019-01-04            0.034285
3 2019-01-07            0.064893
4 2019-01-08           -0.046477


In [18]:
import yfinance as yf

In [19]:
jpmc = yf.download(['JPM'],start = '2010-01-1',end='2025-01-01')
sp500 = yf.download(['^GSPC'],start = '2010-01-1',end='2025-01-01')

/tmp/ipykernel_932/4128732890.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  jpmc = yf.download(['JPM'],start = '2010-01-1',end='2025-01-01')
[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_932/4128732890.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  sp500 = yf.download(['^GSPC'],start = '2010-01-1',end='2025-01-01')
[*********************100%***********************]  1 of 1 completed


In [20]:
jpmc.isnull().sum().sum()

np.int64(0)

In [21]:
jpmc.duplicated().sum()

np.int64(0)

In [22]:
jpmc.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 3774 entries, 2010-01-04 to 2024-12-31
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   (Close, JPM)   3774 non-null   float64
 1   (High, JPM)    3774 non-null   float64
 2   (Low, JPM)     3774 non-null   float64
 3   (Open, JPM)    3774 non-null   float64
 4   (Volume, JPM)  3774 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 176.9 KB


In [23]:
jpmc['returns'] = jpmc['Close']['JPM'].pct_change(fill_method=None)

In [24]:
#volume ratio
volume = jpmc['Volume']['JPM']
jpmc['volume ratio'] = (volume/ volume.rolling(window=20).mean()).shift(1)

In [25]:
#20-day rolling return
jpmc['20-day rolling return'] = jpmc['returns'].rolling(window=20).mean().shift(1)

In [26]:
delta = jpmc['Close']['JPM'].diff()
gain = delta.clip(lower=0).rolling(14).mean()
loss = (-delta.clip(upper=0)).rolling(14).mean()
RSI = 100 - (100 / (1 + gain/loss))
jpmc['RSI'] = RSI.shift(1)

In [27]:
sp500['SP500 returns'] = sp500['Close']['^GSPC'].pct_change(fill_method=None)

In [28]:
jpmc_clean = pd.DataFrame({
    "returns": jpmc['returns'],
    "Volume Ratio": jpmc['volume ratio'],
    "Rolling Returns": jpmc['20-day rolling return'],
    "RSI": jpmc['RSI']
})

In [29]:
jpmc_clean['target'] = (jpmc_clean['returns'] > 0).astype(int).shift(-1)

In [30]:
jpmc_clean['SP500 Returns'] = sp500['SP500 returns']

In [31]:
jpmc_with_sentiment = jpmc_clean.merge(daily_sentiment, left_index=True, right_on='date', how='left')

In [32]:
jpmc_with_sentiment['weighted_sentiment'] = jpmc_with_sentiment['weighted_sentiment'].fillna(0)
print(jpmc_with_sentiment.shape)
print(jpmc_with_sentiment['weighted_sentiment'].isna().sum())

(3774, 8)
0


In [33]:
jpmc_with_sentiment.drop(columns=['date'], inplace=True)

In [34]:
jpmc_with_sentiment.columns.tolist()

['returns',
 'Volume Ratio',
 'Rolling Returns',
 'RSI',
 'target',
 'SP500 Returns',
 'weighted_sentiment']

In [35]:
print(jpmc_clean.shape, jpmc_clean.columns.tolist())
print(jpmc_with_sentiment.shape, jpmc_with_sentiment.columns.tolist())

(3774, 6) ['returns', 'Volume Ratio', 'Rolling Returns', 'RSI', 'target', 'SP500 Returns']
(3774, 7) ['returns', 'Volume Ratio', 'Rolling Returns', 'RSI', 'target', 'SP500 Returns', 'weighted_sentiment']


In [37]:
from sklearn.preprocessing import StandardScaler

In [40]:
features = jpmc_with_sentiment.drop('target', axis=1)
target = jpmc_with_sentiment['target']